# Aura White Studio - artwork to game-ready textured 3D (Colab)

One picture in, a **UV-textured GLB** out (albedo + normal map) - the original artwork's own pixels on every surface it can see,
generated side views for the far side. Runs on a free **T4** (use *Standard* quality) or an **A100/L4** (*High* / *Ultra*).

**Runtime -> Change runtime type -> GPU** before you start.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

## 1. Get the project
Upload `aura-white.zip` (the project you downloaded). If you also have the model repository zips
(`Hunyuan3D-2_1-main.zip`, `InstantMesh-main.zip`, `zero123plus-main.zip`, `nvdiffrast-main.zip`) upload those too - otherwise step 3 downloads them from GitHub.

In [ ]:
from google.colab import files
up = files.upload()          # choose aura-white.zip (+ optional repo zips)

In [ ]:
import glob, os, subprocess
subprocess.run("unzip -q -o aura-white.zip", shell=True, check=True)
%cd aura-white
!pip install -q -e ".[recommended]"

## 2. Extras
Texturing accelerators + the heavy models' Python packages. Nothing here needs a compiler.
If a later step complains that Zero123++ cannot load, run the *pinned* variant instead (`requirements-studio-pinned.txt`) and restart the runtime.

In [ ]:
!pip install -q -r requirements-studio.txt

## 3. Model repositories
Uses the zips you uploaded above when present; otherwise downloads the sources from GitHub (model **weights** download automatically on first use).

In [ ]:
import glob
zips = [z for z in glob.glob("../*.zip") + glob.glob("*.zip") if "aura-white" not in z]
if zips:
    !python -m aura_white setup --zip {" ".join(zips)}
else:
    !python -m aura_white setup

In [ ]:
!python -m aura_white doctor

*(Optional)* NVIDIA's nvdiffrast rasteriser makes baking faster. Aura White's own PyTorch rasteriser is used if you skip this or if the install fails.

In [ ]:
# !pip install -q --no-build-isolation ./third_party/nvdiffrast-main

## 4. Your artwork
A single character/object on a plain or transparent background works best. Concept art, renders and photos are all fine.

In [ ]:
art = files.upload()
ART = next(iter(art))
from IPython.display import Image, display
display(Image(ART, width=320))

## 5. Generate
`--quality`: `draft` (1K, ~1 min) - `standard` (2K, T4-friendly) - `high` (4K, needs ~16 GB+) - `ultra` (200k faces).
`--geometry auto` picks Hunyuan3D-2.1 when it fits in GPU memory, else InstantMesh, else the built-in model.
Add `--multiview off` to skip the side-view generator (and its non-commercial weights licence).

In [ ]:
QUALITY = "standard"   # draft | standard | high | ultra
!python -m aura_white studio "{ART}" --quality {QUALITY} -o studio_out

In [ ]:
import glob
from IPython.display import Image, display
stem = os.path.splitext(ART)[0]
for name in ("preview", "registration", "side_views"):
    for f in glob.glob(f"studio_out/{stem}/asset_{name}.png"):
        print(name); display(Image(f))

## 6. Download
The folder contains `asset.glb` (open in Blender / any glTF viewer), `asset.obj + .mtl`, `asset_albedo.png`, `asset_normal.png` and `asset_report.json`
(the report lists which model made the geometry, how well the artwork matched the mesh, and any warnings).

In [ ]:
!cd studio_out && zip -qr ../studio_out.zip . && cd ..
files.download("studio_out.zip")

### If something goes wrong
* `no CUDA GPU` in `doctor` -> Runtime -> Change runtime type -> GPU.
* Out of memory -> `QUALITY = "draft"` or `--geometry instantmesh`; each model is loaded, used and freed in turn.
* Zero123++ fails to load -> `!pip install -q -r requirements-studio-pinned.txt`, restart runtime, rerun from step 1.
* `report.json -> notes` explains every fallback that was taken (unusable xatlas -> built-in unwrapper, etc.).